# Volume 10, Chapter 3: Weak convergence and Donsker's theorem (Python companion)

Exercises first, solutions at the end. Invariance in action (one limit for many step laws), the arcsine law and its surprises, a Skorokhod embedding built by hand, the law of the iterated logarithm and how slowly it shows itself, and a computer science application: the **Kolmogorov–Smirnov test** of random number generators, including the infamous RANDU. Requires `numpy` (and `matplotlib` for the optional pictures).

In [ ]:
import math
import numpy as np
rng = np.random.default_rng(30)

def kolmogorov_cdf(x, terms=100):
    # P(sup|B_t - t B_1| <= x) = 1 - 2 sum (-1)^(k-1) exp(-2 k^2 x^2)
    if x <= 0:
        return 0.0
    k = np.arange(1, terms + 1)
    return float(1 - 2 * np.sum((-1.0)**(k - 1) * np.exp(-2 * k**2 * x**2)))

## Exercises

**P3.1 (★) One limit, many walks.** For three step laws with mean $0$ and variance $1$ (Rademacher $\pm1$; uniform on $[-\sqrt3,\sqrt3]$; $E-1$ with $E$ exponential of mean $1$), simulate $20000$ walks of $n=1000$ steps and record $M_n=\max_{k\le n}S_k/\sqrt n$. Corollary 3.14 says that $M_n\Rightarrow|Z|$. Compare the empirical distribution functions with $P(|Z|\le x)=\operatorname{erf}(x/\sqrt2)$ by their largest gap. Which walk is slowest to converge, and why might that be?

**P3.2 (★) The arcsine law.** (a) For the coin-tossing walk with $2n=20$ steps, enumerate all $2^{20}$ paths and check that the number of positive edges equals $2k$ with probability $u_{2k}u_{2n-2k}$, where $u_{2m}=\binom{2m}m4^{-m}$. (b) For $2n=10^4$, simulate $10^4$ walks and compare the fraction of time positive with the arcsine distribution function $\frac2\pi\arcsin\sqrt x$. (c) Two players toss a fair coin $10^4$ times. What is the probability that one of them leads more than $90\%$ of the time? Guess before computing.

**P3.3 (★★) A Skorokhod embedding by hand.** The law $\mu$ that puts $\frac23$ on $-1$ and $\frac13$ on $2$ is centred with variance $2$. (a) Simulate Brownian motion (Gaussian steps of variance $\mathrm dt=10^{-4}$) until it leaves $(-1,2)$, $10^4$ times. Check that $B_\tau$ has law $\mu$ and that $\mathbb E\tau\approx2$ (Proposition 3.10). (b) Where does the small bias come from? Which way does it go?

**P3.4 (★★) The iterated logarithm, slowly.** For $20$ Rademacher walks of length $10^7$, plot or tabulate $\max_{10^k\le n<10^{k+1}}S_n/\sqrt{2n\log\log n}$ for $k=2,\dots,6$. Theorem 3.19 says the $\limsup$ is $1$. How close do you get? Why is $\log\log$ hopeless to see in a simulation? (Compute $\log\log n$ for $n=10^7$ and for $n=10^{80}$.)

**P3.5 (★★★) CS application: testing random number generators.** Given $u_1,\dots,u_n$ in $[0,1)$, the Kolmogorov–Smirnov statistic is $D_n=\sup_t|F_n(t)-t|$, computed from the sorted sample as $\max_i\max\bigl(\frac in-u_{(i)},u_{(i)}-\frac{i-1}n\bigr)$. If the $u_i$ are i.i.d. uniform, $\sqrt nD_n$ converges in law to the Kolmogorov distribution (the remark after Theorem 3.17), and the $p$-value of an observed $d$ is about $1-K(\sqrt nd)$.

(a) Implement the test and check that under numpy's generator the $p$-values of $1000$ independent tests (with $n=1000$) are themselves uniform. (b) Test the "generator" $u^{1.02}$: how large must $n$ be before the test notices the bias, and why does the threshold scale like $\delta^{-2}$, where $\delta$ is the largest gap between the two distribution functions? (c) **RANDU**, used on IBM mainframes in the 1960s and 70s: $x_{k+1}=65539\,x_k\bmod2^{31}$, $u_k=x_k/2^{31}$. Run the test on the $u_k$ themselves. Then show that $x_{k+2}=6x_{k+1}-9x_k\pmod{2^{31}}$, and run the test on the fractional part of $u_{k+2}-6u_{k+1}+9u_k$. What is the moral?

## Solutions

### P3.1
**How to find it.** Generate the steps as an $(m,n)$ array, take cumulative sums and the maximum along each row (the maximum over $k\le n$ includes $S_0=0$ only if you add it; with $n=1000$ that makes no difference). The largest gap between an empirical distribution function and a continuous $F$ is attained at a sample point, so compare at the sorted sample.

In [ ]:
def max_gap(sample, F):
    x = np.sort(sample)
    m = len(x)
    Fx = F(x)
    i = np.arange(1, m + 1)
    return max(np.max(i / m - Fx), np.max(Fx - (i - 1) / m))

erf = np.vectorize(math.erf)
F_absZ = lambda x: erf(np.maximum(x, 0) / math.sqrt(2))
m, n = 20000, 1000
laws = {
    "Rademacher": lambda size: rng.choice([-1.0, 1.0], size=size),
    "uniform": lambda size: rng.uniform(-math.sqrt(3), math.sqrt(3), size=size),
    "exponential-1": lambda size: rng.exponential(1.0, size=size) - 1,
}
for name, draw in laws.items():
    M = np.zeros(m)
    for start in range(0, m, 5000):          # in blocks, to save memory
        S = np.cumsum(draw((5000, n)), axis=1)
        M[start:start + 5000] = np.maximum(S.max(axis=1), 0) / math.sqrt(n)
    print(f"{name:14s} E M_n = {M.mean():.4f}  (limit {math.sqrt(2/math.pi):.4f})   max gap = {max_gap(M, F_absZ):.4f}")

All three gaps are small, a few hundredths, and the means are all a little below $\sqrt{2/\pi}\approx0.798$: a walk looked at only at integer times misses the highest points of the continuous path between them, so its maximum is biased downwards, by an amount of order $n^{-1/2}$ (about $0.58/\sqrt n$ for Gaussian steps, where $0.58\approx-\zeta(\frac12)/\sqrt{2\pi}$; Siegmund's book in the chapter's spirit, *Sequential Analysis*, 1985, Chapter X, has the computation). The Rademacher walk has the biggest gap for a different reason: its maximum takes only the values $k/\sqrt n$, so its distribution function is a staircase with steps of height about $\frac1{\sqrt n}\cdot2\varphi$, which no continuous law can follow closely. The uniform walk, smooth and symmetric, is the best. Same limit, different speeds: that is all Donsker's theorem promises.

### P3.2
**How to find it.** (a) Enumerate the paths with bit tricks or with `itertools.product`; an edge from $S_{k-1}$ to $S_k$ is positive when $S_{k-1}+S_k>0$ (Section 3.4). (b) Vectorise over the walks. (c) "One player leads more than $90\%$ of the time" means the fraction of positive time is below $0.1$ or above $0.9$; by the arcsine law this has probability about $2\cdot\frac2\pi\arcsin\sqrt{0.1}$.

In [ ]:
from itertools import product
from fractions import Fraction

two_n = 20
counts = np.zeros(two_n + 1, dtype=np.int64)
steps = np.array(list(product([-1, 1], repeat=two_n)), dtype=np.int8)   # 2^20 x 20
S = np.cumsum(steps, axis=1, dtype=np.int16)
prev = np.hstack([np.zeros((S.shape[0], 1), dtype=np.int16), S[:, :-1]])
pos = ((prev + S) > 0).sum(axis=1)
counts = np.bincount(pos, minlength=two_n + 1)
u = lambda m2: Fraction(math.comb(m2, m2 // 2), 2**m2)
ok = all(Fraction(int(counts[2 * k]), 2**two_n) == u(2 * k) * u(two_n - 2 * k) for k in range(two_n // 2 + 1))
print("odd counts all zero:", counts[1::2].sum() == 0, "  exact match with u_2k u_2n-2k:", ok)
print("P(0 positive edges) =", counts[0] / 2**two_n, "  P(10 positive edges) =", counts[10] / 2**two_n)
del steps, S, prev

In [ ]:
two_n, walks = 10**4, 10**4
frac = np.empty(walks)
for start in range(0, walks, 1000):
    S = np.cumsum(rng.choice(np.array([-1, 1], dtype=np.int16), size=(1000, two_n)), axis=1, dtype=np.int32)
    prev = np.hstack([np.zeros((1000, 1), dtype=np.int32), S[:, :-1]])
    frac[start:start + 1000] = ((prev + S) > 0).mean(axis=1)
F_arc = lambda x: 2 / np.pi * np.arcsin(np.sqrt(np.clip(x, 0, 1)))
print("max gap with the arcsine law:", round(max_gap(frac, F_arc), 4))
p_lead = np.mean((frac < 0.1) | (frac > 0.9))
print(f"P(one player leads > 90% of the time): simulated {p_lead:.3f}, arcsine law {2*F_arc(0.1):.3f}")

(a) The match is exact: the discrete arcsine law of Lemma 3.16. The two extreme values, $0$ and $20$ positive edges, are the *most* likely ones, each with probability $u_{20}\approx0.176$, while a perfect $10$–$10$ split has probability $u_{10}^2\approx0.061$.

(c) About $0.41$. Most people guess something like $0.01$. In a fair game, the typical situation is that one player is ahead almost all the time; a fair share of the lead is the *least* likely outcome. Feller's book (Chapter III) has a famous table with numbers like these.

### P3.3
**How to find it.** Advance all the paths that have not yet exited by one Gaussian step at a time, and freeze each one at its exit. With $p=P(B_\tau=2)$, the martingale $B$ gives $2p-(1-p)=0$, so $p=\frac13$, and the martingale $B_t^2-t$ gives $\mathbb E\tau=\mathbb EB_\tau^2=\frac23+\frac43=2$.

In [ ]:
dt, paths = 1e-4, 10000
B = np.zeros(paths); t = np.zeros(paths); alive = np.ones(paths, bool)
sd = math.sqrt(dt)
while alive.any():
    idx = np.flatnonzero(alive)
    B[idx] += sd * rng.standard_normal(idx.size)
    t[idx] += dt
    alive[idx] = (B[idx] > -1) & (B[idx] < 2)
hit2 = B >= 2
print(f"P(B_tau = 2) = {hit2.mean():.3f} +- {hit2.std()/math.sqrt(paths):.3f} (exact 1/3)")
print(f"E tau = {t.mean():.3f} +- {t.std()/math.sqrt(paths):.3f} (exact 2)")
print(f"mean overshoot past the boundary: {np.mean(np.where(hit2, B - 2, -1 - B)):.4f}")

The exit law and the mean time agree with $\mu$ and with $2$ up to the Monte Carlo error printed after $\pm$ (one standard error) and a small systematic bias. (b) The simulated path is only looked at on the grid $\mathrm dt\,\Z$, so it exits *late*, after overshooting the boundary by a few $\sqrt{\mathrm dt}$ (the mean overshoot is about $0.58\sqrt{\mathrm dt}\approx0.006$, the same constant as in P3.1). Since $\mathbb E\tau=\mathbb EB_\tau^2$ also for the discretised walk, the overshoot inflates $\mathbb E\tau$ slightly, by roughly $2\cdot\frac23\cdot0.006+2\cdot2\cdot\frac13\cdot0.006\approx0.016$. Repeating with $\mathrm dt=10^{-6}$ shrinks it tenfold.

### P3.4
**How to find it.** Generate each walk in blocks of $10^6$ steps and keep, for each decade, the largest ratio seen. The table is the honest answer: the iterated logarithm is a statement about astronomically long times.

In [ ]:
N, walks = 10**7, 20
decades = [(10**k, 10**(k + 1)) for k in range(2, 7)]
best = np.full((walks, len(decades)), -np.inf)
for w in range(walks):
    level = 0
    for start in range(0, N, 10**6):
        S = level + np.cumsum(rng.choice(np.array([-1, 1], dtype=np.int8), size=10**6), dtype=np.int64)
        level = S[-1]
        nn = np.arange(start + 1, start + 10**6 + 1)
        for j, (a, b) in enumerate(decades):
            lo, hi = max(a, start + 1), min(b, start + 10**6 + 1)
            if lo < hi:
                sl = slice(lo - start - 1, hi - start - 1)
                r = S[sl] / np.sqrt(2 * nn[sl] * np.log(np.log(nn[sl])))
                best[w, j] = max(best[w, j], r.max())
for j, (a, b) in enumerate(decades):
    print(f"n in [10^{int(math.log10(a))}, 10^{int(math.log10(b))}):  median of max ratio {np.median(best[:, j]):.3f},  largest {best[:, j].max():.3f}")
print("log log 1e7 =", round(math.log(math.log(1e7)), 3), "   log log 1e80 =", round(math.log(math.log(1e80)), 3))

The medians of the decade maxima sit well below $1$ (between about $0.3$ and $0.8$), and only the luckiest of the $20$ walks get near or above $1$: over a single decade the walk does not have time to reach its eventual $\limsup$, and the convergence in the theorem is with respect to $\log\log n$, which grows from about $2.8$ at $n=10^7$ to only $5.2$ at $n=10^{80}$, a number of steps larger than the number of atoms in the observable universe. The law of the iterated logarithm is a theorem one proves, not one that one sees; simulations show its *scale*, $\sqrt{n\log\log n}$, only roughly.

### P3.5
**How to find it.** (a) A $p$-value of a continuous statistic is uniform under the null hypothesis, so the $p$-values of many tests should pass a KS test themselves. (b) For $u^{1.02}$ the distribution function is $t^{1/1.02}$, and the largest gap $\delta=\max_t|t^{1/1.02}-t|$ is about $0.0072$. The statistic $\sqrt nD_n$ is then about $\sqrt n\,\delta$ plus a fluctuation of order one, so the test sees the bias once $\sqrt n\,\delta$ is a few units: $n$ of order $\delta^{-2}$, here of order $10^5$. (c) $65539=2^{16}+3$, so $65539^2=2^{32}+6\cdot2^{16}+9\equiv6\cdot65539-9\pmod{2^{31}}$, which gives $x_{k+2}=6x_{k+1}-9x_k$ modulo $2^{31}$.

In [ ]:
def ks_test(u):
    u = np.sort(np.asarray(u, dtype=float))
    n = len(u)
    i = np.arange(1, n + 1)
    D = max(np.max(i / n - u), np.max(u - (i - 1) / n))
    return D, 1 - kolmogorov_cdf(math.sqrt(n) * D)

# (a) p-values under a good generator
pvals = np.array([ks_test(rng.random(1000))[1] for _ in range(1000)])
print("(a) KS test of 1000 p-values: D = %.4f, p = %.3f" % ks_test(pvals))

# (b) a slightly biased generator
t = np.linspace(0, 1, 100001)
delta = np.max(np.abs(t**(1 / 1.02) - t))
print(f"(b) delta = {delta:.4f}")
for n in [10**3, 10**4, 10**5, 10**6]:
    D, p = ks_test(rng.random(n)**1.02)
    print(f"    n = {n:>7}: sqrt(n) D = {math.sqrt(n)*D:6.2f}, p = {p:.2g}")

In [ ]:
def randu(n, seed=1):
    x = np.empty(n, dtype=np.int64); x[0] = seed
    for k in range(1, n):
        x[k] = (65539 * x[k - 1]) % 2**31
    return x

x = randu(10**5 + 2, seed=4321)
u = x / 2**31
print("(c) RANDU, the u_k themselves: D = %.4f, p = %.3f" % ks_test(u[:10**5]))
print("    x_{k+2} = 6 x_{k+1} - 9 x_k mod 2^31 for all k:", np.all((x[2:] - 6 * x[1:-1] + 9 * x[:-2]) % 2**31 == 0))
w = np.mod(u[2:] - 6 * u[1:-1] + 9 * u[:-2], 1.0)
w = np.where(w > 0.5, w - 1, w)          # rounding: the true value is an integer
print("    largest |frac(u_{k+2} - 6u_{k+1} + 9u_k)| =", np.abs(w).max())
v = rng.random(10**5 + 2)
print("    the same statistic for a good generator: D = %.4f, p = %.3f" % ks_test(np.mod(v[2:] - 6 * v[1:-1] + 9 * v[:-2], 1.0)))
print("    ... and for RANDU: D = %.4f, p = %.3g" % ks_test(np.mod(u[2:] - 6 * u[1:-1] + 9 * u[:-2] + 0.5, 1.0)))

(a) The $p$-values are uniform, as they should be. (b) The bias of $u^{1.02}$ is invisible at $n=10^3$, borderline around $n=10^4$ (where $\sqrt n\,\delta\approx0.7$, so the outcome depends on the luck of the sample) and unmistakable from $10^5$ on: a goodness-of-fit test detects a deviation $\delta$ only with about $\delta^{-2}$ samples, the $\sqrt n$ of Donsker's theorem in a statistician's coat.

(c) RANDU passes the one-dimensional test: its $p$-value is an ordinary one, as for a good generator, because its outputs *are* very evenly spread over $[0,1)$. But $u_{k+2}-6u_{k+1}+9u_k$ is always an integer (up to rounding), so its fractional part, which for a good generator is uniform, is a point mass (shifted to $\frac12$ in the last line, to keep the floating point rounding away from the wrap-around at $0$), and the test rejects it with a $p$-value that underflows to $0$. Geometrically, every triple $(u_k,u_{k+1},u_{k+2})$ lies on one of $15$ parallel planes in the unit cube, as Marsaglia showed in 1968 for all such multiplicative generators ("Random numbers fall mainly in the planes"). The moral: a test sees only what it measures. A test suite like TestU01 runs hundreds of statistics, most of them reduced, through a central limit or a Donsker theorem, to a known limit law, for exactly that reason.